In [119]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.linear_model import Lasso


from collections import namedtuple
Dataset = namedtuple('Dataset', ['X_train', 'X_test', 'y_train', 'y_test'])
Params = namedtuple('Params', ['min_samples_leaf', 'max_features', 'max_depth', 'oob_score'])
Data = namedtuple('Data', ['X','y'])
parameters = Params(3, 0.3, 25, False)


import sys
sys.path.append('../library')

import classification_rf
from importlib import reload
reload(classification_rf)

import rf_wrapper_functions as rfwrap

np.random.seed(24)

In [120]:
'''
Import Patient data, map strings to integers, one hot encode non ordinal features (race), 
drop unimportant or output leaking features, 
'''

df = pd.read_excel("../Input_Data/TGCA Cancer Classification/mmc1.xlsx", sheet_name="TCGA-CDR")

chosen_types = ["KIRC", "PAAD", "BRCA", "BLCA", "LIHC", "HNSC", "THCA", "SKCM"] #Chosen for minimal missing data
df = df[df["type"].isin(chosen_types)].copy()

na_tokens = ["[Not Available]", "[Not Applicable]", "[Unknown]",
             "[Discrepancy]", "[Not Evaluated]"]
df = df.replace(na_tokens, np.nan)
df = df[df["Redaction"].isna()].copy()

df = df.dropna(subset=["OS", "OS.time"])

#Map strings to integer values

df["gender"] = df["gender"].replace({"MALE": 0, "FEMALE": 1})

stage_map = {
    "Stage 0":    0,
    "Stage I":    1,   "Stage IA": 1,   "Stage IB": 1.3,
    "I/II NOS":   1.5,
    "Stage II":   2,   "Stage IIA": 2,  "Stage IIB": 2.3, "Stage IIC": 2.6,
    "Stage III":  3,   "Stage IIIA": 3, "Stage IIIB": 3.3, "Stage IIIC": 3.6,
    "Stage IV":   4,   "Stage IVA": 4,  "Stage IVB": 4.3, "Stage IVC": 4.6,
    "Stage X":    np.nan,
}

df["ajcc_pathologic_tumor_stage"] = df["ajcc_pathologic_tumor_stage"].replace(stage_map)

#One hot encoding for race. If race is nan then all will be 0
race_dummies = pd.get_dummies(df["race"], prefix="race", dummy_na=False).astype(int)
df = pd.concat([df.drop(columns=["race"]), race_dummies], axis=1)

#These columns are kind of useless due to missing data for certain types
df = df.drop(columns=[
    #Useless or mostly missing data
    "clinical_stage", 
    "histological_grade", 
    "histological_type", 
    "menopause_status", 
    "Redaction",
    "Unnamed: 0",
    "birth_days_to",


    #Possible outcome leaks
    "new_tumor_event_type",
    "new_tumor_event_site",
    "new_tumor_event_site_other",
    "new_tumor_event_dx_days_to",
    "treatment_outcome_first_course",
    "margin_status",
    "residual_tumor",
    "last_contact_days_to", 
    "death_days_to", 
    "cause_of_death",
    "vital_status",
    "tumor_status",
    "DSS", "DSS.time",
    "DFI", "DFI.time",
    "PFI", "PFI.time",
    ])

print(df["type"].value_counts())

type
BRCA    1091
KIRC     537
HNSC     527
THCA     507
SKCM     455
BLCA     408
LIHC     375
PAAD     185
Name: count, dtype: int64


In [121]:
'''Grab Aneuploidy Score and Mutation Count'''

df_biomarkers = pd.read_csv("../Input_Data/TGCA Cancer Classification/cBioportal_data.tsv", delimiter='\t')

# Sample type code is the suffix after the last '-' in Sample ID
# ('01' = primary tumor, '06' = metastatic)
df_biomarkers["sample_type_code"] = df_biomarkers["Sample ID"].str.split("-").str[-1]

priority = {"01": 0, "06": 1}  # prefer primary tumor when a patient has both
df_biomarkers["priority"] = df_biomarkers["sample_type_code"].map(priority).fillna(99)

df_biomarkers = (
    df_biomarkers
    .sort_values("priority")
    .drop_duplicates(subset="Patient ID", keep="first")
)

#Important Columns
df_biomarkers_subset = df_biomarkers[["Patient ID", "Sample ID", "Aneuploidy Score", "Mutation Count"]]

df_biomarkers_subset = df_biomarkers_subset.rename(columns={"Patient ID": "bcr_patient_barcode"})

In [122]:
'''
Grab General: TP53 PanCan Pathways (6 genes) from cbioportal.org
'''

df_genomics_data = pd.read_csv('../Input_Data/TGCA Cancer Classification/sample_matrix.txt', sep='\t')

#Split Study ID and Sample ID columns
df_genomics_data[["studyID", "sampleId"]] = df_genomics_data["studyID:sampleId"].str.split(':', expand=True)

#Drop unimportant columns
df_genomics_data = df_genomics_data.drop(columns=["studyID:sampleId", "studyID", "Altered"])

#Rename column for merge
df_genomics_data = df_genomics_data.rename(columns={"sampleId": "Sample ID"})

#Merge genomics onto biometric dataframe
df_genomics_biometric = pd.merge(df_biomarkers_subset, df_genomics_data, on='Sample ID', how="left")

#Merge Biometric and Genomic dataframe onto patient info dataframe
df = pd.merge(df, df_genomics_biometric, on='bcr_patient_barcode', how="left")

#FYI there are duplicates due to sample/patient when merging, but the dataset is large enough that this is negligable

In [123]:
#make type last column
df["type"] = df.pop("type")
type_map = {name: idx for idx, name in enumerate(chosen_types)}
df["type_encoded"] = df["type"].map(type_map)

#182.5 days in six months
time = 730
df["Two_Year_Death"] = np.where(((df["OS"] == 1) & (df["OS.time"] < time)), 1, 0)

#remove patients with OS 0 and time under 6 months
df_two_year_death_trimmed = df[~((df["OS"] == 0) & (df["OS.time"] < time))]

# Drop specified columns and convert to NumPy array
X = df.drop(columns=["bcr_patient_barcode", "OS","type"]).to_numpy()
y = df[["Two_Year_Death"]].to_numpy().astype(int)

print(df_two_year_death_trimmed.groupby('type')['Two_Year_Death'].value_counts())
#print(df_two_year_death_trimmed["type"].value_counts())
#print(df["type"].value_counts())

#Send to data storage
df.to_csv("../Input_Data/TGCA Cancer Classification/TGCA_Processed_(Untrimmed_OS).csv", index=False)
df_two_year_death_trimmed.to_csv("../Input_Data/TGCA Cancer Classification/TGCA_Processed.csv", index=False)

type  Two_Year_Death
BLCA  1                 142
      0                 141
BRCA  0                 598
      1                  39
HNSC  0                 233
      1                 161
KIRC  0                 362
      1                  83
LIHC  0                 144
      1                  91
PAAD  1                  89
      0                  38
SKCM  0                 289
      1                  75
THCA  0                 313
      1                   4
Name: count, dtype: int64


In [124]:
#Grab mutation count and aneuploidy score

import requests

BASE_URL = "https://www.cbioportal.org/api"

type_to_study = {
    "KIRC": "kirc_tcga_pan_can_atlas_2018",
    "PAAD": "paad_tcga_pan_can_atlas_2018",
    "BRCA": "brca_tcga_pan_can_atlas_2018",
    "BLCA": "blca_tcga_pan_can_atlas_2018",
    "LIHC": "lihc_tcga_pan_can_atlas_2018",
    "HNSC": "hnsc_tcga_pan_can_atlas_2018",
    "THCA": "thca_tcga_pan_can_atlas_2018",
    "SKCM": "skcm_tcga_pan_can_atlas_2018",
}

WANTED_KEYWORDS = ["mutation count", "aneuploidy score"]

all_rows = []
for cancer_type, study_id in type_to_study.items():
    resp = requests.get(
        f"{BASE_URL}/studies/{study_id}/clinical-data",
        params={"clinicalDataType": "SAMPLE", "projection": "DETAILED"},
    )
    resp.raise_for_status()
    data = resp.json()

    kept = [
        row for row in data
        if any(kw in row["clinicalAttribute"]["displayName"].strip().lower()
               for kw in WANTED_KEYWORDS)
    ]

    for row in kept:
        all_rows.append({
            "bcr_patient_barcode": row["patientId"],
            "attribute": row["clinicalAttribute"]["displayName"],
            "value": row["value"],
        })
    print(f"{cancer_type}: fetched {len(kept)} matching records out of {len(data)} total")

biomarkers_long = pd.DataFrame(all_rows)
biomarkers = biomarkers_long.pivot_table(
    index="bcr_patient_barcode", columns="attribute",
    values="value", aggfunc="first"
).reset_index()

#biomarkers.to_csv("cbioportal_biomarkers.csv", index=False)
#print(biomarkers.head())
#biomarkers = pd.read_csv("cbioportal_biomarkers.csv")
#df = df.merge(biomarkers, on="bcr_patient_barcode", how="left")

KIRC: fetched 885 matching records out of 9234 total
PAAD: fetched 344 matching records out of 3462 total
BRCA: fetched 2107 matching records out of 19348 total
BLCA: fetched 811 matching records out of 7788 total
LIHC: fetched 727 matching records out of 7031 total
HNSC: fetched 1025 matching records out of 9852 total
THCA: fetched 954 matching records out of 8917 total
SKCM: fetched 874 matching records out of 6925 total


In [125]:
#Grab mutation count and aneuploidy score

import requests

BASE_URL = "https://www.cbioportal.org/api"

type_to_study = {
    "KIRC": "kirc_tcga_pan_can_atlas_2018",
    "PAAD": "paad_tcga_pan_can_atlas_2018",
    "BRCA": "brca_tcga_pan_can_atlas_2018",
    "BLCA": "blca_tcga_pan_can_atlas_2018",
    "LIHC": "lihc_tcga_pan_can_atlas_2018",
    "HNSC": "hnsc_tcga_pan_can_atlas_2018",
    "THCA": "thca_tcga_pan_can_atlas_2018",
    "SKCM": "skcm_tcga_pan_can_atlas_2018",
}

WANTED_KEYWORDS = ["mutation count", "aneuploidy score"]

all_rows = []
for cancer_type, study_id in type_to_study.items():
    resp = requests.get(
        f"{BASE_URL}/studies/{study_id}/clinical-data",
        params={"clinicalDataType": "SAMPLE", "projection": "DETAILED"},
    )
    resp.raise_for_status()
    data = resp.json()

    kept = [
        row for row in data
        if any(kw in row["clinicalAttribute"]["displayName"].strip().lower()
               for kw in WANTED_KEYWORDS)
    ]

    for row in kept:
        all_rows.append({
            "bcr_patient_barcode": row["patientId"],
            "attribute": row["clinicalAttribute"]["displayName"],
            "value": row["value"],
        })
    print(f"{cancer_type}: fetched {len(kept)} matching records out of {len(data)} total")

biomarkers_long = pd.DataFrame(all_rows)
biomarkers = biomarkers_long.pivot_table(
    index="bcr_patient_barcode", columns="attribute",
    values="value", aggfunc="first"
).reset_index()

#biomarkers.to_csv("cbioportal_biomarkers.csv", index=False)
#print(biomarkers.head())
#biomarkers = pd.read_csv("cbioportal_biomarkers.csv")
#df = df.merge(biomarkers, on="bcr_patient_barcode", how="left")

KIRC: fetched 885 matching records out of 9234 total
PAAD: fetched 344 matching records out of 3462 total
BRCA: fetched 2107 matching records out of 19348 total
BLCA: fetched 811 matching records out of 7788 total
LIHC: fetched 727 matching records out of 7031 total
HNSC: fetched 1025 matching records out of 9852 total
THCA: fetched 954 matching records out of 8917 total
SKCM: fetched 874 matching records out of 6925 total


In [126]:
'''
This imported some set of genes that claude thought was good. 
I went back later to use instead a recommended list from the website itself
'''
import requests

BASE_URL = "https://www.cbioportal.org/api"

type_to_study = {
    "KIRC": "kirc_tcga_pan_can_atlas_2018",
    "PAAD": "paad_tcga_pan_can_atlas_2018",
    "BRCA": "brca_tcga_pan_can_atlas_2018",
    "BLCA": "blca_tcga_pan_can_atlas_2018",
    "LIHC": "lihc_tcga_pan_can_atlas_2018",
    "HNSC": "hnsc_tcga_pan_can_atlas_2018",
    "THCA": "thca_tcga_pan_can_atlas_2018",
    "SKCM": "skcm_tcga_pan_can_atlas_2018",
}

# Recurrent pan-cancer driver genes -- adjust this list as you like
GENES_OF_INTEREST = ["TP53", "PIK3CA", "KRAS", "PTEN", "APC", "EGFR", "BRAF"]

# --- Step 1: resolve Hugo symbols -> Entrez IDs (mutation endpoint needs Entrez IDs) ---
entrez_ids = {}
for gene in GENES_OF_INTEREST:
    resp = requests.get(f"{BASE_URL}/genes/{gene}")
    resp.raise_for_status()
    entrez_ids[gene] = resp.json()["entrezGeneId"]
print("Entrez IDs:", entrez_ids)

all_flag_rows = []
for cancer_type, study_id in type_to_study.items():
    # --- Step 2: get the full patient list for this study (so non-mutated
    # patients still get an explicit 0, not just a missing row) ---
    samples_resp = requests.get(f"{BASE_URL}/studies/{study_id}/samples")
    samples_resp.raise_for_status()
    samples = samples_resp.json()
    all_patients = sorted({s["patientId"] for s in samples})

    # --- Step 3: fetch mutations for our gene set in this study ---
    mut_resp = requests.post(
        f"{BASE_URL}/molecular-profiles/{study_id}_mutations/mutations/fetch",
        json={
            "entrezGeneIds": list(entrez_ids.values()),
            "sampleListId": f"{study_id}_all",
        },
        headers={"Content-Type": "application/json"},
    )
    mut_resp.raise_for_status()
    mutations = mut_resp.json()

    # map entrezGeneId back to Hugo symbol
    entrez_to_symbol = {v: k for k, v in entrez_ids.items()}

    # set of (patientId, gene) pairs that have >=1 mutation
    mutated_pairs = {
        (m["patientId"], entrez_to_symbol[m["entrezGeneId"]])
        for m in mutations
    }

    # build a 0/1 row per patient per gene
    for patient in all_patients:
        row = {"bcr_patient_barcode": patient}
        for gene in GENES_OF_INTEREST:
            row[f"{gene}_mut"] = int((patient, gene) in mutated_pairs)
        all_flag_rows.append(row)

    print(f"{cancer_type}: {len(all_patients)} patients, "
          f"{len(mutations)} mutation records in gene set")

mutation_flags = pd.DataFrame(all_flag_rows)
#mutation_flags.to_csv("cbioportal_mutation_flags.csv", index=False)
#print(mutation_flags.head())

#mutation_flags = pd.read_csv("cbioportal_mutation_flags.csv")
#df = df.merge(mutation_flags, on="bcr_patient_barcode", how="left")

#make type last column
df["type"] = df.pop("type")
type_map = {name: idx for idx, name in enumerate(chosen_types)}
df["type_encoded"] = df["type"].map(type_map)
df.to_csv("../Input_Data/TGCA Cancer Classification/TGCA_Processed_Claude_Genes.csv", index=False)

Entrez IDs: {'TP53': 7157, 'PIK3CA': 5290, 'KRAS': 3845, 'PTEN': 5728, 'APC': 324, 'EGFR': 1956, 'BRAF': 673}
KIRC: 512 patients, 41 mutation records in gene set
PAAD: 184 patients, 241 mutation records in gene set
BRCA: 1084 patients, 856 mutation records in gene set
BLCA: 411 patients, 425 mutation records in gene set
LIHC: 372 patients, 155 mutation records in gene set
HNSC: 523 patients, 594 mutation records in gene set
THCA: 499 patients, 303 mutation records in gene set
SKCM: 442 patients, 527 mutation records in gene set


In [127]:
# Drop specified columns and convert to NumPy array
X = df.drop(columns=["bcr_patient_barcode", "OS","type"]).to_numpy()
y = df["OS"].to_numpy().astype(int)


In [128]:
X_train, X_test, y_train, y_test = rfwrap.split_by_source(X, y, test_size=0.2, random_state=24)
dataset = Dataset(X_train, X_test, y_train, y_test)
data = Data(X,y)